# ARIMA and SARIMA Forecasting
This notebook trains and evaluates ARIMA and SARIMA models on the daily revenue data.
- Uses `statsmodels.tsa.arima.model.ARIMA`
- Uses `statsmodels.tsa.statespace.sarimax.SARIMAX`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings("ignore")
plt.style.use('ggplot')

### 1. Load Data

In [ ]:
df = pd.read_csv("../data/cleaned/cleaned_retail.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

sales = df.groupby('InvoiceDate')['TotalPrice'].sum().reset_index()
sales.columns = ['ds', 'y']
sales = sales.resample('D', on='ds').sum().reset_index()
sales['y'] = sales['y'].fillna(0)
sales.set_index('ds', inplace=True)

print(f"Total days of data: {len(sales)}")
sales.head()

### 2. ARIMA Model

In [ ]:
# Fit ARIMA model
model_arima = ARIMA(sales['y'], order=(5,1,0))
result_arima = model_arima.fit()

# Forecast future 30 days
forecast_arima = result_arima.forecast(steps=30)

# In-sample prediction to calculate RMSE on last 30 days
in_sample_arima = result_arima.predict(start=len(sales)-30, end=len(sales)-1)
rmse_arima = np.sqrt(mean_squared_error(sales['y'][-30:], in_sample_arima))
print(f'ARIMA RMSE (last 30 days): {rmse_arima:.2f}')

plt.figure(figsize=(10,5))
plt.plot(sales.index[-60:], sales['y'][-60:], label='Actual')
plt.plot(sales.index[-30:], in_sample_arima, label='ARIMA In-Sample Forecast', linestyle='--')
plt.plot(pd.date_range(start=sales.index[-1] + pd.Timedelta(days=1), periods=30), forecast_arima, label='ARIMA Future Forecast', linestyle=':')
plt.title('ARIMA Demand Forecasting')
plt.legend()
plt.show()

### 3. SARIMA Model

In [ ]:
# Fit SARIMA model
model_sarima = SARIMAX(sales['y'], order=(1,1,1), seasonal_order=(1,1,0,12))
result_sarima = model_sarima.fit(disp=False)

# Forecast future 30 days
forecast_sarima = result_sarima.forecast(steps=30)

# In-sample prediction to calculate RMSE on last 30 days
in_sample_sarima = result_sarima.predict(start=len(sales)-30, end=len(sales)-1)
rmse_sarima = np.sqrt(mean_squared_error(sales['y'][-30:], in_sample_sarima))
print(f'SARIMA RMSE (last 30 days): {rmse_sarima:.2f}')

plt.figure(figsize=(10,5))
plt.plot(sales.index[-60:], sales['y'][-60:], label='Actual')
plt.plot(sales.index[-30:], in_sample_sarima, label='SARIMA In-Sample Forecast', linestyle='--')
plt.plot(pd.date_range(start=sales.index[-1] + pd.Timedelta(days=1), periods=30), forecast_sarima, label='SARIMA Future Forecast', linestyle=':')
plt.title('SARIMA Demand Forecasting')
plt.legend()
plt.show()